# Export to TFLite
Convert the trained Keras model to a quantized TFLite model for on-device mobile inference.

In [ ]:
import pathlib
import tensorflow as tf

MODEL_DIR = '../models/saved_model'
OUT_PATH = pathlib.Path('../models/freshness_model.tflite')

model = tf.keras.models.load_model(MODEL_DIR)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

OUT_PATH.write_bytes(tflite_model)
print(f'Wrote {OUT_PATH} ({OUT_PATH.stat().st_size / 1024:.1f} KB)')

## Sanity check the converted model on a few validation images

In [ ]:
import numpy as np

interpreter = tf.lite.Interpreter(model_path=str(OUT_PATH))
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

with open('../models/class_names.txt') as f:
    class_names = f.read().splitlines()

def predict(img_array):
    img_array = img_array.astype(input_details[0]['dtype'])
    interpreter.set_tensor(input_details[0]['index'], img_array[None, ...])
    interpreter.invoke()
    preds = interpreter.get_tensor(output_details[0]['index'])[0]
    return class_names[np.argmax(preds)], float(np.max(preds))